In [2]:
import cv2
import datetime
import pandas as pd
import numpy as np
from keras.models import load_model

In [9]:
# Load models from Kaggle dataset path
age_model = load_model("/kaggle/input/age-gender-models/age_model.h5", compile=False)
gender_model = load_model("/kaggle/input/age-gender-models/gender_model.h5", compile=False)

In [10]:
# Load a sample video (upload your own to a Kaggle dataset)
video_path = "/kaggle/input/testvideo1/videoplayback.mp4"
cap = cv2.VideoCapture(video_path)

In [11]:
log = []

In [14]:
while True:
    ret, frame = cap.read()
    if not ret:
        break
    try:
        face = cv2.resize(frame, (200, 200)) / 255.0
        face = np.expand_dims(face, axis=0)
        age = int(age_model.predict(face)[0][0])
        gender_pred = gender_model.predict(face)
        gender = "Male" if gender_pred[0][0] > 0.5 else "Female"
        time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        if age < 13 or age > 60:
            message = "Not allowed"
            color = (0, 0, 255)
        else:
            message = "Allowed"
            color = (0, 255, 0)

        # Optional: annotate frame (disabled in headless Kaggle, but left for completeness)
        # cv2.putText(frame, message, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        # cv2.rectangle(frame, (10, 10), (220, 220), color, 2)

        log.append({"Age": age, "Gender": gender, "Time": time, "Status": message})

    except Exception as e:
        print("Frame skipped due to error:", e)

cap.release()

In [15]:
# Save log to CSV
df = pd.DataFrame(log)
df.to_csv("/kaggle/working/horror_ride_log.csv", index=False)
print("Log saved to /kaggle/working/horror_ride_log.csv")

Log saved to /kaggle/working/horror_ride_log.csv
